# Document Question Answering with RAG
### A step-by-step walkthrough of a Retrieval-Augmented Generation pipeline

This notebook builds a RAG system from scratch and demonstrates every stage:

1. Document Ingestion
2. Text Chunking
3. Embedding Creation
4. Vector Database (similarity search)
5. Query Processing
6. Context Retrieval
7. Answer Generation

We use the lightweight, fully-offline default backends (TF-IDF embeddings +
extractive answer generation) so this notebook runs with zero API keys and
zero model downloads. Swapping in `sentence-transformers` or an LLM backend
is a one-line change, shown at the bottom.

In [1]:
import sys
sys.path.insert(0, '..')
from src.document_loader import load_document
from src.chunker import chunk_text
from src.embedder import get_embedder
from src.vector_store import get_vector_store
from src.generator import get_generator
from src.rag_pipeline import RagPipeline

ModuleNotFoundError: No module named 'pypdf'

## Stage 1: Document Ingestion
Load the sample document and inspect the raw text.

In [ ]:
text = load_document('../sample_docs/sample.txt')
print(f"Loaded {len(text)} characters, {len(text.split())} words")
print(text[:400], '...')

Loaded 2240 characters, 354 words
Retrieval-Augmented Generation, or RAG, is a technique that combines information
retrieval with text generation. Instead of relying only on what a language model
learned during training, a RAG system first searches a knowledge base for relevant
information and then uses that information to generate an answer. This makes the
answers more accurate and lets the model work with private or up-to-date d ...


## Stage 2: Text Chunking
Split the document into overlapping word-bounded chunks.

In [ ]:
chunks = chunk_text(text, chunk_size=80, overlap=15, source='sample.txt')
print(f"Produced {len(chunks)} chunks\n")
for c in chunks[:2]:
    print(f"--- Chunk {c.id} ---")
    print(c.text[:200], '...\n')

Produced 6 chunks

--- Chunk 0 ---
Retrieval-Augmented Generation, or RAG, is a technique that combines information retrieval with text generation. Instead of relying only on what a language model learned during training, a RAG system  ...

--- Chunk 1 ---
never trained on. The RAG pipeline usually has four main stages. First, documents are ingested and converted into raw text. Second, that text is split into smaller chunks so that retrieval can be more ...



## Stage 3: Embedding Creation
Convert each chunk into a numeric vector. We use TF-IDF here -- no downloads required.

In [ ]:
embedder = get_embedder('tfidf')
texts = [c.text for c in chunks]
embeddings = embedder.fit_transform(texts)
print(f"Embedding matrix shape: {embeddings.shape}  (chunks x vocabulary features)")

Embedding matrix shape: (6, 308)  (chunks x vocabulary features)


## Stage 4: Vector Database
Store the embeddings for fast similarity search.

In [ ]:
store = get_vector_store('numpy')
store.build(embeddings, chunks)
print(f"Indexed {len(store.chunks)} chunks into the vector store")

Indexed 6 chunks into the vector store


## Stages 5 & 6: Query Processing + Context Retrieval
Embed a question with the *same* embedder, then find the most similar chunks.

In [ ]:
question = "What are the four main stages of a RAG pipeline?"
query_vec = embedder.transform([question])[0]
results = store.search(query_vec, top_k=2)

for chunk, score in results:
    print(f"[score={score:.3f}] chunk {chunk.id}: {chunk.text[:150]}...\n")

[score=0.261] chunk 0: Retrieval-Augmented Generation, or RAG, is a technique that combines information retrieval with text generation. Instead of relying only on what a lan...

[score=0.227] chunk 1: never trained on. The RAG pipeline usually has four main stages. First, documents are ingested and converted into raw text. Second, that text is split...



## Stage 7: Answer Generation
Pass the retrieved chunks to a generator to produce the final answer.

The default `extractive` generator needs no model at all. Swap in `huggingface` (local flan-t5) or `anthropic` (Claude API) for higher-quality generated prose -- see the last section.

In [ ]:
generator = get_generator('extractive')
context_chunks = [chunk for chunk, score in results]
answer = generator.generate(question, context_chunks)
print(answer)

Based on the document, here is the most relevant passage:

"Retrieval-Augmented Generation, or RAG, is a technique that combines information retrieval with text generation. Instead of relying only on what a language model learned during training, a RAG system first searches a knowledge base for relevant information and then uses that information to generate an answer. This makes the answers more accurate and lets the model work with private or up-to-date data that it was never trained on. The RAG pipeline usually has four main stages. First, documents are ingested"

(source: sample.txt, chunk #0)


## Putting it all together: `RagPipeline`
The `RagPipeline` class wraps stages 1-7 into a single object so you don't have to wire them by hand every time.

In [ ]:
pipeline = RagPipeline(embedder_name='tfidf', generator_name='extractive', chunk_size=80, overlap=15, top_k=3)
stats = pipeline.index_document('../sample_docs/sample.txt')
print(stats)

{'num_chunks': 6, 'embedding_dim': 308}


## Evaluating retrieval quality
A simple Hit Rate @ K check: for a set of (question, expected keyword) pairs, did the retrieved chunks actually contain the keyword?

In [ ]:
from src.evaluate import hit_rate_at_k

qa_pairs = [
    ("What are the four stages of RAG?", "chunk"),
    ("What is hybrid search?", "BM25"),
    ("Why is RAG better than fine-tuning?", "hallucination"),
]
report = hit_rate_at_k(pipeline, qa_pairs, k=3)
print(f"Hit Rate @3: {report['hit_rate']}")
for d in report['details']:
    print(f"  [{'PASS' if d['hit'] else 'FAIL'}] {d['question']}")

Hit Rate @3: 1.0
  [PASS] What are the four stages of RAG?
  [PASS] What is hybrid search?
  [PASS] Why is RAG better than fine-tuning?


## Try it on your own document
Replace the path below with your own PDF, resume, or notes file.

```python
my_pipeline = RagPipeline()
my_pipeline.index_document('../sample_docs/your_file.pdf')
print(my_pipeline.answer('Your question here')['answer'])
```
